# Day 8 - LLM foundations & API integration practical tasks
***

These tasks cover calling OpenAI API's and prompting.

## Explore how different model parameters and different prompts influence the response.
#### Here are some guidelines and advice you should consider:
* You can add as many cells as you need for each task.
* Use Markdown cells for your conclusions.
* For tasks that do not specify a model, you may choose any of the two available models: `gpt-5.6-luna` / `gpt-5.4-mini`.
* Within each comparison task, change only the model, prompt, or parameter specified in the instructions. Keep everything else - including the model, prompt, and other settings - the same across calls so you can compare the results fairly. For example, when testing different `temperature` values, use the same model, prompt, and output token limit (`max_output_tokens` for the Responses API or `max_completion_tokens` for the Chat Completions API).
* You can use both - [Responses API and Chat Completions API](https://medium.com/@sanathgoutham/openai-responses-api-vs-chat-completions-what-actually-changed-eeee86aff216) - to complete these tasks, but some model parameters are not supported by Responses API.
* LLMs understand [Markdown](https://www.markdownguide.org/cheat-sheet/) formatting in prompts. For example, wrapping text in double asterisks `(**important text**)` marks it as **bold** and can help signal emphasis.
* To copy text from a Markdown cell, open it in edit mode by clicking the pencil icon or double-clicking the cell.
* Optionally, paste API responses into a Markdown cell for better readability.
* Be creative and experiment.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

env_path = Path.cwd() / ".env"

if not env_path.exists():
    env_path = Path.cwd() / "day 8" / ".env"

load_dotenv(env_path, override=True)

key = os.getenv("OPENAI_API_KEY", "").strip()

print("ENV found:", env_path.exists())
print("Key loaded:", bool(key))

ENV found: True
Key loaded: True


In [2]:
client = OpenAI(
    base_url="https://ds-ai-internship.openai.azure.com/openai/v1/",
    api_key=key
)

print("Client configured")

Client configured


In [4]:
import os

key = os.getenv("OPENAI_API_KEY", "")

print("Key loaded:", bool(key))
print("Key length:", len(key))
print("API endpoint:", client.base_url)
print("Client key matches ENV:", client.api_key == key)

Key loaded: True
Key length: 102
API endpoint: https://ds-ai-internship.openai.azure.com/openai/v1/
Client key matches ENV: True


#### 1. Write a Python script to call the OpenAI API and generate a response to the prompt below.

Prompt: `"What is artificial intelligence?"`

Use model `gpt-5.6-luna`.

In [3]:
response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {"role": "user", "content": "What is artificial intelligence?"}
    ],
    reasoning_effort="none",
    max_completion_tokens=300
)

print(response.choices[0].message.content)
print("Total tokens:", response.usage.total_tokens)

Artificial intelligence (AI) is the field of creating computer systems that can perform tasks normally requiring human intelligence, such as:

- Understanding language
- Recognizing images or speech
- Learning from data
- Solving problems
- Making predictions or decisions

AI systems can be designed with explicit rules, trained on examples using machine learning, or built with modern techniques such as neural networks and generative models. Examples include virtual assistants, recommendation systems, self-driving technologies, fraud detection, and chatbots.
Total tokens: 114


#### 2. Call the OpenAI API with different models

Prompt `"Explain the concept of machine learning to a 10-year-old."`

`model`: `gpt-5.6-luna` and `gpt-5.4-mini`.

Compare the outputs from each model. 

You can evaluate the differences in:
* Clarity of explanation.
* Simplicity of language (suitable for a 10-year-old).
* Creativity in the response.
* Response time.

In [4]:
import time

prompt = "Explain the concept of machine learning to a 10-year-old."

start = time.perf_counter()

response_luna = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[{"role": "user", "content": prompt}],
    max_completion_tokens=350
)

luna_time = time.perf_counter() - start

print(response_luna.choices[0].message.content)
print("Response time:", round(luna_time, 2), "seconds")

Machine learning is a way of teaching computers to make guesses or decisions by showing them lots of examples instead of giving them every rule.

For example, imagine teaching a computer to recognize cats:

1. You show it many pictures labeled “cat” and “not a cat.”
2. The computer looks for patterns, such as pointy ears, whiskers, and certain shapes.
3. When it sees a new picture, it uses those patterns to guess whether there is a cat.

The computer does not “think” like a person. It finds patterns in the examples it was given. The more useful examples it gets, the better its guesses may become.

Machine learning is used for things like:

- Recommending videos or music
- Recognizing voices
- Helping cars notice objects
- Sorting emails into spam
- Translating languages

It can still make mistakes, especially if its examples are limited, confusing, or unfair.
Response time: 6.3 seconds


In [5]:
start = time.perf_counter()

response_mini = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": prompt}],
    max_completion_tokens=350
)

mini_time = time.perf_counter() - start

print(response_mini.choices[0].message.content)
print("Response time:", round(mini_time, 2), "seconds")

Machine learning is a way to teach computers to **learn from examples** instead of being told **every single rule**.

### Simple idea:
Imagine you want a computer to tell the difference between a **cat** and a **dog**.

Instead of writing:
- “If it has pointy ears, maybe it’s a cat”
- “If it barks, maybe it’s a dog”

You show the computer **lots of pictures** of cats and dogs.  
After seeing many examples, the computer starts to notice patterns on its own.

### Like a kid learning:
It’s kind of like how you learned to:
- recognize apples and oranges
- know when a face is smiling
- tell different animals apart

You didn’t memorize a giant list of rules first. You learned by **seeing examples**.

### What machine learning does:
A machine learning program:
1. Looks at lots of data
2. Finds patterns
3. Uses those patterns to make guesses about new things

### Example:
If it has learned from many photos of cats, it can look at a new photo and say:
- “I think this is a cat!”

It won’t always

Both models explained machine learning in a way that a 10-year-old could understand.

Luna gave a clear step-by-step explanation using pictures of cats. It also mentioned that machine learning can make mistakes.

Mini used a more relatable comparison with how children learn from examples. I found its explanation a little more engaging, although it was longer.

The biggest difference was speed: Luna took 6.30 seconds, while Mini took only 2.58 seconds. In this test, I preferred Mini because it was faster and its explanation felt more suitable for a child.

In [7]:
def ask(prompt, model="gpt-5.4-mini", max_tokens=300, system=None, **options):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    return client.chat.completions.create(
        model=model,
        messages=messages,
        max_completion_tokens=max_tokens,
        reasoning_effort="none",
        **options
    )

#### 3. Call the OpenAI API with different `temperature` values

Prompt: `"Write a two sentence story about a robot learning to cook."`

`temperature` values: `0`, `0.7` and `1.5`. Generate three responses at each temperature.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare the outputs.

Tip: With Chat Completions API you can use parameter `n=3` to generate three different responses to your prompt with one call.

In [8]:
prompt = "Write a two sentence story about a robot learning to cook."

for temp in [0, 0.7, 1.5]:
    response = ask(
        prompt,
        temperature=temp,
        max_tokens=150,
        n=3
    )

    print(f"\nTEMPERATURE = {temp}")
    for i, choice in enumerate(response.choices, 1):
        print(f"\nStory {i}:")
        print(choice.message.content)


TEMPERATURE = 0

Story 1:
The robot carefully studied recipes, measuring flour and stirring soup with precise, whirring motions until the kitchen smelled like warm bread. When it finally served dinner, it proudly announced that it had learned the secret ingredient was not code, but patience.

Story 2:
The robot carefully studied recipes, its metal fingers clinking as it chopped vegetables and stirred a bubbling pot with precise, practiced motions. When it finally served dinner, the robot’s sensors glowed with pride as everyone agreed that its first homemade soup was delicious.

Story 3:
The robot carefully studied recipes, its metal fingers clinking as it chopped vegetables and stirred a bubbling pot with precise, practiced motions. When it finally served dinner, the robot’s sensors glowed with pride as everyone agreed that its first homemade soup was delicious.

TEMPERATURE = 0.7

Story 1:
The robot carefully studied recipes, measured ingredients with mechanical precision, and stirre

At temperature 0, the stories were very similar, and two of them were exactly the same.

At temperature 0.7, the stories became more varied, although they still followed a similar idea.

At temperature 1.5, I noticed more creative descriptions and unusual expressions. For example, one story described the robot as having a tiny metal heart.

I think temperature 0 is better for consistent results, while higher temperatures are more useful for creative writing.

#### 4. Call the OpenAI API with different `max_output_tokens`/`max_completion_tokens` values.

Prompt: `"Explain the concept of machine learning in simple terms."`

`max_output_tokens` (Response API call)/`max_completion_tokens` (Chat Completions API call) values: `50`, `100` and `200`.

Use `gpt-5.4-mini` for this task.

Compare the length and detail of the responses.

In [9]:
prompt = "Explain the concept of machine learning in simple terms."

for limit in [50, 100, 200]:
    response = ask(prompt, max_tokens=limit)

    print(f"\nTOKEN LIMIT = {limit}")
    print(response.choices[0].message.content)
    print("Used tokens:", response.usage.completion_tokens)
    print("Finish reason:", response.choices[0].finish_reason)


TOKEN LIMIT = 50
Machine learning is a way for computers to **learn patterns from data** instead of being told every rule step by step.

### Simple idea
Imagine teaching a computer by showing it many examples:

- If you show it lots of photos of
Used tokens: 50
Finish reason: length

TOKEN LIMIT = 100
Machine learning is a way for computers to **learn from examples** instead of being told every rule step by step.

### Simple idea
Imagine teaching a child to recognize cats:
- You show many pictures of cats and dogs.
- Over time, the child notices patterns, like whiskers, ears, and face shape.
- Then the child can guess whether a new picture is a cat or a dog.

Machine learning works similarly:
- You give the computer lots of data.
- It
Used tokens: 100
Finish reason: length

TOKEN LIMIT = 200
Machine learning is a way for computers to learn from examples instead of being told every rule explicitly.

### Simple idea:
If you show a computer lots of examples, it can start to notice patter

With a limit of 50 tokens, the explanation stopped in the middle of a sentence. The same happened with 100 tokens, although the answer contained more details.

With a limit of 200 tokens, the model completed its explanation using 183 tokens. The finish reason was "stop" instead of "length".

This shows that a low token limit can cut off useful information, while a higher limit gives the model enough space to finish its answer.

#### 5. Call the OpenAI API with different `top_p` values.

Prompt: `"Write a list of 5 completely random and unusual hobbies that no one has ever heard of."`

`top_p` values: `0.3`, `0.7`, and `1.0`.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare response variation as you adjust `top_p`.

In [10]:
prompt = "Write a list of 5 completely random and unusual hobbies that no one has ever heard of."

for p in [0.3, 0.7, 1.0]:
    response = ask(prompt, top_p=p, max_tokens=300)

    print(f"\nTOP_P = {p}")
    print(response.choices[0].message.content)


TOP_P = 0.3
Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow spotting** — tracking and sketching the shapes clouds cast on the ground at different times of day.  
2. **Typewriter sound collecting** — recording and cataloging the distinct noises of different typewriters.  
3. **Pebble balancing in wind** — trying to stack stones in outdoor places where the breeze makes it extra difficult.  
4. **Map margin doodling** — creating tiny illustrations only in the blank borders of old maps.  
5. **Spoon reflection photography** — taking photos of distorted scenes reflected in the curved backs of spoons.

TOP_P = 0.7
Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow spotting** — tracking the shapes and movement of shadows cast by clouds on hills, buildings, or water.
2. **Doorway acoustics testing** — standing in different doorframes to compare how voices and sounds change in each one.
3. **Pebble alphabet collecting** — searching for rocks that 

At top_p = 0.3, the hobbies were unusual but still somewhat realistic, like photographing reflections in spoons.

At 0.7, the ideas became more playful, including "sock archaeology".

At 1.0, some suggestions were much stranger, such as writing apologies to chairs.

I noticed that higher top_p allowed more unexpected ideas, although some concepts were repeated across the responses.

#### 6. Call the OpenAI API with different `frequency_penalty` values.

Prompt: `"Write a list of funny one-liner jokes about cats."`

`frequency_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the repetition of joke formats changes as you increase the `frequency_penalty`.

In [11]:
prompt = "Write a list of funny one-liner jokes about cats."

for penalty in [0, 0.5, 1.0]:
    response = ask(
        prompt,
        frequency_penalty=penalty,
        max_tokens=350
    )

    print(f"\nFREQUENCY PENALTY = {penalty}")
    print(response.choices[0].message.content)


FREQUENCY PENALTY = 0
Sure — here are some funny one-liner cat jokes:

1. My cat is not spoiled; he’s just better cared for than I am.  
2. Cats are like little furry landlords who demand rent in treats.  
3. I asked my cat what he wanted for dinner, and he stared at me like I’d asked for a memoir.  
4. My cat doesn’t ignore me — he just selectively responds to royalty.  
5. Cats: proof that gravity is just a suggestion.  
6. My cat trained me to open doors, refill bowls, and apologize for existing.  
7. I got a cat for companionship, and now I live in his apartment.  
8. My cat’s favorite hobby is knocking things over and calling it “interior design.”  
9. Cats don’t have owners — they have staff.  
10. My cat acts like he pays the bills, but all he brings home is chaos.  
11. I wanted a pet with personality, and my cat gave me six.  
12. My cat’s idea of helping is sitting directly on whatever I’m trying to do.  
13. Cats are the only animals that can look offended by their own refl

At frequency_penalty = 0, I noticed repeated phrases like "My cat" in several jokes.

I tested higher frequency penalties to see whether the wording would become more varied. The penalty is intended to discourage repeated tokens, but it does not necessarily prevent similar jokes or ideas.

The experiment showed that controlling word repetition and generating genuinely different ideas are not exactly the same thing.

#### 7. Call the OpenAI API with different `presence_penalty` values.

Prompt: `"Suggest ten names for a new line of eco-friendly cleaning products."`

`presence_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the diversity of generated names changes as you increase the `presence_penalty`.

In [12]:
prompt = "Suggest ten names for a new line of eco-friendly cleaning products."

for penalty in [0, 0.5, 1.0]:
    response = ask(
        prompt,
        presence_penalty=penalty,
        max_tokens=350
    )

    print(f"\nPRESENCE PENALTY = {penalty}")
    print(response.choices[0].message.content)


PRESENCE PENALTY = 0
Here are ten name ideas for an eco-friendly cleaning product line:

1. **Green Gleam**
2. **PureSprout**
3. **EcoShine**
4. **EarthKind Clean**
5. **FreshLeaf Home**
6. **NaturAnew**
7. **SustainSpark**
8. **CleanNest**
9. **Verdant Vigor**
10. **BloomBright Clean**

If you want, I can also generate names that sound more **premium**, **modern**, **playful**, or **scientific/natural**.

PRESENCE PENALTY = 0.5
Here are ten name ideas for an eco-friendly cleaning product line:

1. **Green Gleam**
2. **PureNest**
3. **EarthSpritz**
4. **EcoShine**
5. **Fresh Fern**
6. **Verdant Clean**
7. **NatureNook**
8. **CleanLeaf**
9. **BioBrite**
10. **Sprout & Spark**

If you want, I can also give you names that sound more **premium**, **playful**, **minimalist**, or **science-based**.

PRESENCE PENALTY = 1.0
Here are ten name ideas for an eco-friendly cleaning product line:

1. **PureSprout**
2. **GreenGleam**
3. **EarthEssence Clean**
4. **EcoShine**
5. **FreshLeaf Home**
6. 

I noticed that some product names were repeated even when I increased presence_penalty. For example, "Green Gleam" and "EcoShine" appeared with both 0 and 0.5.

Higher presence penalty encourages the model to introduce new words, but it doesn't guarantee completely different ideas between separate responses.

I think this parameter can help with variety, but I would still need to check the suggestions for duplicates.

#### 8. Use the prompt "*Explain the importance of data preprocessing in machine learning.*" with the following system prompts:
1. "You are a university professor teaching a class of beginners."
2. "You are a technical expert explaining to a team of engineers."
3. "You are a comedian explaining this concept in a funny way."

Observe how the tone and style of the response change based on the system prompt.

In [13]:
prompt = "Explain the importance of data preprocessing in machine learning."

personas = {
    "University Professor": "You are a university professor teaching beginners. Explain clearly and use simple examples.",
    "ML Engineer": "You are an experienced machine learning engineer. Give a precise technical explanation.",
    "Comedian": "You are a comedian. Explain the topic using humor and jokes."
}

for name, system_prompt in personas.items():
    response = ask(
        prompt,
        system=system_prompt,
        max_tokens=300
    )

    print(f"\nPERSONA: {name}")
    print(response.choices[0].message.content)


PERSONA: University Professor
Data preprocessing is very important in machine learning because raw data is usually messy, incomplete, and inconsistent. A machine learning model can only learn well if the data is prepared properly.

### Why it matters

1. **Improves model accuracy**
   - If the data has errors, missing values, or strange outliers, the model may learn the wrong patterns.
   - Clean data helps the model make better predictions.

2. **Makes data usable**
   - Many algorithms cannot work directly with text, missing values, or different data scales.
   - Preprocessing turns raw data into a format the model can understand.

3. **Reduces noise**
   - Real-world data often contains irrelevant information or mistakes.
   - Removing noise helps the model focus on useful patterns.

4. **Helps algorithms perform better**
   - Some models are sensitive to the scale of numbers.
   - Example: if one feature is “age” and another is “income,” income values are much larger. Scaling help

The system prompts changed how the model explained the same topic.

The professor gave a structured explanation with clear examples, while the comedian used jokes to make the topic more entertaining.

I also tested the role of an ML engineer to get a more technical explanation.

I noticed that system prompts are useful for adapting the same information to different audiences without changing the main question.

#### 9. Write a Zero-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!".

Use it in your API call.

In [14]:
review = "The product is amazing, and I love using it every day!"

system_prompt = (
    "Classify this product review as Positive, Negative, or Neutral. "
    "Return only the sentiment label."
)

zero_shot = ask(
    review,
    system=system_prompt,
    max_tokens=100
)

print("Zero-shot prediction:")
print(zero_shot.choices[0].message.content)

Zero-shot prediction:
Positive


#### 10. Write a Few-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!"

Use it in your API call.

Observe how the model's performance changes and compare the response to Zero-Shot output.

In [15]:
response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": "The delivery was terrible."
        },
        {
            "role": "assistant",
            "content": "Negative"
        },
        {
            "role": "user",
            "content": "Everything works perfectly."
        },
        {
            "role": "assistant",
            "content": "Positive"
        },
        {
            "role": "user",
            "content": "The package arrived on Monday."
        },
        {
            "role": "assistant",
            "content": "Neutral"
        },
        {
            "role": "user",
            "content": review
        }
    ],
    reasoning_effort="none",
    max_completion_tokens=100
)

print("Zero-Shot:", zero_shot.choices[0].message.content)
print("Few-Shot:", response.choices[0].message.content)

Zero-Shot: Positive
Few-Shot: Positive


Both Zero-Shot and Few-Shot prompting classified the review as Positive.

In the Zero-Shot test, the model received only the instructions and the review. In the Few-Shot test, I also provided three examples with the expected sentiment labels.

The result didn't change in this case, but Few-Shot prompting can help the model understand the required format and labels. One example isn't enough to prove that either method is more accurate.

#### 11. Optimize input tokens.

Prompt: `"As you know, streaming services and social media platforms use artificial intelligence to recommend movies, shows, and songs to people. These smart recommendation systems work by looking closely at your past viewing habits, what categories you like, and what similar users enjoy watching, and then they predict what you might want to see next. Keeping this background in mind, explain in two simple sentences why a user who loves sci-fi movies suddenly sees a romantic comedy recommended."`

Rewrite the prompt to use as few input tokens as possible while preserving its meaning and the requirement to answer in two simple sentences. The response should remain relevant, clear, and accurate.

Run the original and revised prompts using the same model and settings. Compare their input token counts and assess whether shortening the prompt affected the response quality.

You can use the API response’s usage information to check token counts.

In [16]:
original_prompt = (
    "As you know, streaming services and social media platforms use "
    "artificial intelligence to recommend movies, shows, and songs to people. "
    "These smart recommendation systems work by looking closely at your past "
    "viewing habits, what categories you like, and what similar users enjoy "
    "watching, and then they predict what you might want to see next. "
    "Keeping this background in mind, explain in two simple sentences why "
    "a user who loves sci-fi movies suddenly sees a romantic comedy recommended."
)

short_prompt = (
    "In two simple sentences, explain why an AI recommender might "
    "suggest a romantic comedy to a sci-fi fan based on viewing "
    "history, genre preferences, and similar users."
)

original = ask(original_prompt, max_tokens=200)
shortened = ask(short_prompt, max_tokens=200)

print("ORIGINAL RESPONSE:")
print(original.choices[0].message.content)
print("Input tokens:", original.usage.prompt_tokens)

print("\nSHORTENED RESPONSE:")
print(shortened.choices[0].message.content)
print("Input tokens:", shortened.usage.prompt_tokens)

print("\nTOKENS SAVED:",
      original.usage.prompt_tokens - shortened.usage.prompt_tokens)

ORIGINAL RESPONSE:
The system may have noticed that many people who like sci-fi also enjoy romantic comedies, so it recommends one based on similar users’ preferences. It could also be trying to mix in something different from the user’s usual choices to keep recommendations fresh and interesting.
Input tokens: 94

SHORTENED RESPONSE:
An AI recommender may notice that the sci-fi fan has also watched movies with humor, adventure, or strong character relationships, which often overlap with romantic comedies. It may also see that similar users enjoyed romantic comedies, so it suggests one based on shared patterns in viewing history and preferences.
Input tokens: 39

TOKENS SAVED: 55


The original prompt used 94 input tokens, while the shorter version used only 39, saving 55 tokens.

Both responses explained the recommendation using viewing preferences and similarities between users. The shorter prompt was much more efficient without losing the main idea.

This experiment showed me that a longer prompt doesn't always produce a better answer. Keeping the important details and removing unnecessary background can save tokens.

#### 12. Optimize output tokens.

Prompt: `"Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"`

Rewrite the prompt to request a concise, plausible explanation in 10 output tokens or fewer.

Run the original and revised prompts using the same model and settings. Compare their output token counts and check whether the shorter response still provides a meaningful explanation. Refine your prompt if necessary.

You can use the API response’s usage information to check token counts.

In [18]:
short_prompt = (
    "Why might a sci-fi fan be recommended a romantic comedy? "
    "Answer in 4 English words maximum. "
    "No introduction or formatting."
)

shortened = ask(short_prompt, max_tokens=100)

print("SHORTENED RESPONSE:")
print(shortened.choices[0].message.content)
print("Output tokens:", shortened.usage.completion_tokens)

SHORTENED RESPONSE:
Opposites attract in genres
Output tokens: 9


The original response used 100 output tokens and was cut off before finishing.

My first shorter prompt produced a response using 13 tokens, which was still above the required limit.

I made the instruction stricter by asking for a maximum of four words. The final response was "Opposites attract in genres" and used only 9 tokens.

This showed me that limiting the number of words can help reduce token usage, but I still need to check the actual token count.

#### 13. Expand the given prompt accordingly to achieve the requested output from the list below.

Prompt: `"Explain the importance of exercise." `

Expected output (from three separate prompts):
1. One sentence long
2. Paragraph format
3. Bullet points

In [19]:
prompts = [
    "Explain the importance of exercise in exactly one sentence.",
    "Explain the importance of exercise in one paragraph of 4-5 sentences.",
    "Explain the importance of exercise using 5 bullet points."
]

for i, prompt in enumerate(prompts, 1):
    response = ask(prompt, max_tokens=350)

    print(f"\nFORMAT {i}:")
    print(response.choices[0].message.content)


FORMAT 1:
Exercise is important because it strengthens the body, improves mental health, and reduces the risk of many chronic diseases.

FORMAT 2:
Exercise is important because it helps keep the body healthy and strong by improving heart health, muscle strength, and flexibility. It also supports mental well-being by reducing stress, improving mood, and increasing energy levels. Regular physical activity can lower the risk of many diseases, such as obesity, diabetes, and high blood pressure. In addition, exercise helps people sleep better and maintain a better quality of life overall.

FORMAT 3:
- **Improves physical health:** Regular exercise strengthens the heart, muscles, and bones, and helps reduce the risk of diseases like obesity, diabetes, and high blood pressure.  
- **Boosts mental well-being:** It can lower stress, anxiety, and depression while improving mood and overall emotional health.  
- **Increases energy levels:** Exercise helps the body work more efficiently, so you o

All three responses followed the requested formats.

The first response explained the importance of exercise in a single sentence. The second gave a more detailed explanation in four sentences, while the third organized the information into five bullet points.

I found the bullet-point version easier to read because each benefit was separated clearly. This showed me how a simple change in a prompt can make the same information more suitable for different purposes.

#### 14. Expand this prompt to get more creative responses.

Prompt: `"Explain Artificial Intelligence."`

Expected output (from three separate prompts):
1. Metaphor
2. Story
3. Funny

In [20]:
prompts = [
    "Explain Artificial Intelligence using a simple metaphor.",
    "Explain Artificial Intelligence through a short story about a student.",
    "Explain Artificial Intelligence in a funny way with a joke."
]

for i, prompt in enumerate(prompts, 1):
    response = ask(prompt, max_tokens=350)

    print(f"\nCREATIVE STYLE {i}:")
    print(response.choices[0].message.content)


CREATIVE STYLE 1:
Think of **Artificial Intelligence** like a **really fast apprentice chef** in a kitchen.

- At first, the chef doesn’t know much.
- You show it thousands of recipes, ingredients, and cooking results.
- Over time, it starts noticing patterns:  
  - “These ingredients usually go together.”  
  - “This cooking method makes the dish taste better.”  
  - “If the sauce looks like this, the meal probably needs less salt.”

Soon, it can help cook new dishes by making smart guesses based on what it learned.

So, **AI is a machine that learns from examples and patterns to do tasks that normally need human intelligence**—like recognizing images, understanding language, or making decisions.

CREATIVE STYLE 2:
Maya was a student who loved solving problems, but she often felt overwhelmed by homework, notes, and exam preparation.

One day, her teacher introduced her to an AI study assistant. At first, Maya thought it was just a fancy calculator. But soon she noticed something diff

The three prompts explained AI in very different ways.

The metaphor compared AI to an apprentice chef learning from recipes. The story about Maya showed how AI could help a student with everyday tasks. The humorous version used a joke about improving a neural network.

I liked the cooking metaphor the most because it explained learning from examples in a simple and memorable way. The experiment showed me that creative prompts can make technical topics more accessible.

#### 15. Expand this prompt for different audiences.

Prompt: `"Explain climate change."`

Expected audience (from three separate prompts):
1. Kids
2. General public
3. Politicians

In [21]:
prompts = [
    "Explain climate change to a 10-year-old using simple words.",
    "Explain climate change to the general public in an informative way.",
    "Explain climate change to policymakers, focusing on risks and policy decisions."
]

for i, prompt in enumerate(prompts, 1):
    response = ask(prompt, max_tokens=350)

    print(f"\nAUDIENCE {i}:")
    print(response.choices[0].message.content)


AUDIENCE 1:
Climate change means the Earth is getting warmer over a long time.

Here’s a simple way to think about it:

- The Sun warms the Earth.
- Some heat should go back into space.
- But gases from cars, factories, and power plants trap extra heat like a blanket.
- This makes the planet warmer than it should be.

What can happen because of this?

- Hotter days
- Stronger storms
- Less snow in some places
- Melting ice at the poles
- Animals and plants having a harder time living in their homes

What can we do?

- Save electricity
- Walk, bike, or take the bus when possible
- Recycle
- Plant trees
- Waste less food

So, climate change is like Earth wearing a blanket that is too thick. We need to help Earth cool down a little.

AUDIENCE 2:
Climate change refers to long-term changes in Earth’s average weather patterns. Today, when people say “climate change,” they usually mean the rapid warming of the planet caused mainly by human activities.

## What is happening?
Earth is getting 

The explanations changed depending on the target audience.

For children, the model used simple language and compared greenhouse gases to a blanket trapping heat. The explanation for the general public was intended to be informative, while the version for policymakers focused on economic risks, water supplies, agriculture, and security.

I found the children's version the easiest to understand. This task showed me that specifying the audience helps make the same topic more relevant to different people.

#### 16. Expand this prompt accordingly to instructions below.

Prompt: `"Describe a sunset."`

Expected result (from three separate prompts):

1. Add more details to the prompt, e.g., on what details should it focus on.
2. Constrain the prompt by forbidding it to use words 'sun' and 'light'.
3. Change the output in the style of Shakespeare.

In [23]:
import re

prompts = [
    (
        "Describe a sunset over the sea in detail. "
        "Focus on the changing colors, clouds, "
        "reflections on the water, and atmosphere."
    ),
    (
        "Describe a sunset in exactly 3 sentences. "
        "Do not use the words 'sun' or 'light'."
    ),
    (
        "Describe a sunset in the poetic style of Shakespeare."
    )
]

for i, prompt in enumerate(prompts, 1):
    response = ask(prompt, max_tokens=600)
    answer = response.choices[0].message.content

    print(f"\nVERSION {i}:")
    print(answer)

    if i == 2:
        forbidden = re.findall(
            r"\b(?:sun|light)\b",
            answer,
            re.IGNORECASE
        )
        print("Forbidden words:", forbidden)


VERSION 1:
The sun hangs low over the horizon, turning the sea into a vast sheet of shifting light. Near the sun itself, the sky burns with intense gold and molten orange, as if the clouds have been brushed from within by fire. Higher up, those colors soften into bands of apricot, rose, and pale coral, then gradually deepen into lavender and indigo where the day is giving way to evening.

Clouds drift across the scene in long, wispy streaks and scattered cotton-like clusters. Their undersides glow with warm copper and blush tones, while their edges are rimmed in silver and pale pink. Some clouds appear almost transparent, letting the fading light pass through them in delicate layers; others are denser, their centers tinted with muted plum and gray-blue, giving the sky a sense of depth and movement.

On the water, the sunset is mirrored in a wavering path of light. A bright, trembling ribbon stretches from the sun toward the shore, broken into glittering fragments by the small ripples 

#### 17. Write your observations from tasks 13-16, e.g., how different prompts influence complexity, tone, style and focus of the responses. 

In tasks 13–16, I noticed that changing the prompt can make a big difference even when the topic stays the same.

In Task 13, I controlled the length and structure of the answer. Task 14 showed how metaphors, stories, and humor can make AI easier and more interesting to explain.

In Task 15, the model changed its language depending on the audience. The explanation for children was simple, while the one for policymakers focused more on economic and security risks.

Task 16 was especially interesting because I could control both the details and writing style. The model described the sunset in different ways and successfully avoided the forbidden words "sun" and "light".

Overall, I learned that more specific prompts give me more control over the final response.

#### 18. Generate a Python code with API call.

Write a prompt that generates a Python function to calculate the factorial of a number. Once you have written the prompt, use it to call the API and generate the code. Test the generated code to ensure it calculates the factorial correctly for various inputs.

In [24]:
prompt = """
Write a Python function named factorial(n).

Requirements:
- Calculate the factorial of a non-negative integer.
- Return 1 for n = 0.
- Reject negative numbers with ValueError.
- Use simple Python without external libraries.
- Return only Python code without explanations or Markdown.
"""

response = ask(prompt, max_tokens=400)

generated_code = response.choices[0].message.content
print(generated_code)

def factorial(n):
    if n < 0:
        raise ValueError("n must be a non-negative integer")
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result


In [31]:
def factorial(n):
    if n < 0:
        raise ValueError("n must be a non-negative integer")
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

assert factorial(0) == 1
assert factorial(1) == 1
assert factorial(3) == 6
assert factorial(5) == 120
assert factorial(7) == 5040

try:
    factorial(-1)
    print("Negative input test failed")
except ValueError:
    print("Negative input test passed")

print("All tests passed")

Negative input test passed
All tests passed


#### 19. Provide the following buggy Python code to the API and ask it to fix the errors:

```
def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)
```

Check if the fixed code is working.

In [32]:
buggy_code = """
def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)
"""

response = ask(
    "Fix the errors in this Python code. Return only corrected Python code:\n"
    + buggy_code,
    max_tokens=300
)

fixed_code = response.choices[0].message.content.strip()
fixed_code = fixed_code.removeprefix("```python").removeprefix("```").removesuffix("```").strip()

print(fixed_code)

def add_numbers(a, b):
    return a + b

print(add_numbers(5, 10))


In [33]:
exec(fixed_code)

assert add_numbers(5, 10) == 15
assert add_numbers(2, 3) == 5
assert add_numbers(-5, 3) == -2

print("All tests passed!")

15
All tests passed!


#### 20. Summarize this text to 2-3 sentences by using API call.

"Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.

At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.

Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney."

[<p align="right"> <cite> Source: What is generative AI? </cite></p>](https://www.ibm.com/think/topics/generative-ai)

In [34]:
text = """
Generative AI, sometimes called "gen AI", refers to deep learning models
that can create complex original content such as long-form text, high-quality
images, realistic video or audio and more in response to a user’s prompt or request.

At a high level, generative models encode a simplified representation of their
training data, and then draw from that representation to create new work
that’s similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data.
But over the last decade, they evolved to analyze and generate more complex data
types. This evolution coincided with the emergence of three sophisticated
deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled
models that could generate multiple variations of content in response to a
prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they
are unrecognizable, and then remove the noise to generate original images
in response to prompts.

Transformers (also called transformer models), which are trained on sequenced
data to generate extended sequences of content (such as words in sentences,
shapes in an image, frames of a video or commands in software code).
Transformers are at the core of most of today’s headline-making generative AI
tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney.
"""

prompt = (
    "Summarize the following text in 2-3 clear sentences. "
    "Keep the main idea and the three types of generative models.\n\n"
    + text
)

response = ask(prompt, max_tokens=250)

print(response.choices[0].message.content)

Generative AI uses deep learning models to create new content like text, images, video, and audio by learning patterns from training data and producing similar but not identical outputs. The three main types of generative models are variational autoencoders (VAEs), diffusion models, and transformers, with transformers powering many leading tools like ChatGPT, GPT-4, Copilot, and Midjourney.


#### 21. Translate the following idioms into a language of your choice, ensuring the meaning is preserved (not just word-for-word translation):

1. "It's raining cats and dogs."
2. "Break a leg."
3. "Kill two birds with one stone."
4. "Spill the beans."
5. "A blessing in disguise."

Try to improve the response as much as possible and evaluate the accuracy of the final output.

In [35]:
idioms = """
1. It's raining cats and dogs.
2. Break a leg.
3. Kill two birds with one stone.
4. Spill the beans.
5. A blessing in disguise.
"""

prompt = (
    "Translate these English idioms into natural Russian. "
    "Do not translate them literally. "
    "For each idiom, give a natural Russian equivalent "
    "and explain its meaning in one short sentence.\n\n"
    + idioms
)

response = ask(prompt, max_tokens=500)

print(response.choices[0].message.content)


1. **It's raining cats and dogs.** — **Льёт как из ведра.**  
   Meaning: It is raining very heavily.

2. **Break a leg.** — **Ни пуха ни пера.**  
   Meaning: A way to wish someone good luck, especially before a performance.

3. **Kill two birds with one stone.** — **Убить двух зайцев одним выстрелом.**  
   Meaning: To achieve two things with one action.

4. **Spill the beans.** — **Выдать секрет.**  
   Meaning: To reveal a secret or sensitive information.

5. **A blessing in disguise.** — **Не было бы счастья, да несчастье помогло.**  
   Meaning: Something bad at first turns out to be good in the end.


All five idioms were translated into natural Russian expressions, and their original meanings were preserved.

I especially liked the translations of "Break a leg" and "A blessing in disguise", because translating them word for word would make little sense in Russian.

I checked the meanings and found no significant mistakes. This showed me that AI can translate expressions based on context rather than individual words.

#### 22. Generate a paragraph about a cat without using the letter 'e'/'E'.

Write a prompt that forces the AI to produce a three-sentence story about a cat while strictly forbidding it to use letter 'e'.

Use `gpt-5.4-mini` for this task.

Explain why your prompt is effective.

In [37]:
improved_prompt = f"""
Your previous attempt was:

{story}

It contained 6 instances of the forbidden letter 'e'.

Rewrite the story following these strict rules:
1. Write exactly 3 sentences about a cat.
2. Do not use the letter 'e' or 'E' anywhere.
3. Avoid words ending in '-ed'.
4. Check every word before answering.
5. Return only the corrected story.
"""

response = ask(improved_prompt, max_tokens=300)

story = response.choices[0].message.content

print(story)
print("\nLetter E count:", story.lower().count("e"))
print("Sentence count:", len(re.findall(r"[.!?](?=\s|$)", story)))

A cat sat on a rug, watching rain fall by a bay window.  
It bat at a yarn ball and purred with joy as dusk drew in.  
Soon it curl up, calm and snug, and drift off in a soft nap.

Letter E count: 2
Sentence count: 3


In [38]:
prompt = f"""
Here is your previous story:

{story}

Rewrite it correctly.

Requirements:
- Exactly 3 sentences about a cat.
- ZERO occurrences of the letter E or e.
- Use grammatically correct English.
- Use simple present tense throughout.
- Do not include explanations or headings.
- Check your answer before responding.

Return only the corrected story.
"""

response = ask(prompt, max_tokens=200)
story = response.choices[0].message.content.strip()

e_count = story.lower().count("e")
sentence_count = len(re.findall(r"[.!?](?=\s|$)", story))

print(story)
print("\nLetter E count:", e_count)
print("Sentence count:", sentence_count)
print("Task passed:", e_count == 0 and sentence_count == 3)

A cat sits on a rug and watches rain fall by a bay window.  
It bats at a yarn ball and purrs with joy as dusk draws in.  
Soon it curls up, calm and snug, and drifts off in a soft nap.

Letter E count: 1
Sentence count: 3
Task passed: False


In [39]:
prompt = f"""
Correct this story:

{story}

The word "watches" contains the forbidden letter "e".
Replace it with "looks at" and preserve the rest of the story.

Requirements:
- Exactly 3 grammatically correct sentences.
- No letter "e" or "E" anywhere.
- Return only the corrected story.
"""

response = ask(prompt, max_tokens=200)
story = response.choices[0].message.content.strip()

print(story)
print("\nLetter E count:", story.lower().count("e"))
print("Sentence count:", len(re.findall(r"[.!?](?=\s|$)", story)))
print("Task passed:", story.lower().count("e") == 0 and len(re.findall(r"[.!?](?=\s|$)", story)) == 3)

A cat sits on a rug and looks at rain fall by a bay window.  
It bats at a yarn ball and purrs with joy as dusk draws in.  
Soon it curls up, calm and snug, and drifts off in a soft nap.

Letter E count: 0
Sentence count: 3
Task passed: True


In my first recorded attempt, the story had 3 sentences but still contained 2 forbidden letters.

After improving the prompt, the model fixed the grammar and reduced the number of forbidden letters to 1.

I then pointed out the remaining mistake, and the model corrected it. The final story contained 0 forbidden letters and exactly 3 sentences.

This experiment showed me that LLMs can struggle with strict character-level instructions. Checking the output with Python and giving specific feedback helped me get the correct result.

#### 23. Extract JSON Data from the input text in two different ways.

Input Text: "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."

Extract the information from the text above using two different methods:

1. Write a prompt asking the AI to return a JSON object with the fields `destination`, `start_date`, and `stay_duration`.

2. Instead of putting rules in the prompt, provide a specific JSON schema that defines the field names and their required data types `destination` (the first city), `start_date` (formatted as YYYY-MM-DD), and `stay_duration` (the number of nights as an integer).

Which method yields a more reliable result for a computer program to process, and why?

In [41]:
trip_text = (
    "Our trip to Tokyo starts on June 12th. "
    "We're staying for 5 nights at the Park Hyatt "
    "before heading to Kyoto."
)

prompt = (
    "Extract information from the text as JSON with three fields: "
    "destination (first city), start_date (YYYY-MM-DD), "
    "and stay_duration (integer number of nights). "
    "Assume the year is 2026 because it is not specified. "
    "Return only JSON.\n\n" + trip_text
)

response1 = ask(prompt, max_tokens=200)

print("METHOD 1 - PROMPT:")
print(response1.choices[0].message.content)

METHOD 1 - PROMPT:
{"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5}


In [42]:
import json

schema = {
    "type": "object",
    "properties": {
        "destination": {
            "type": "string",
            "description": "The first city in the trip"
        },
        "start_date": {
            "type": "string",
            "description": "YYYY-MM-DD format. Assume 2026 if year is missing."
        },
        "stay_duration": {
            "type": "integer",
            "description": "Number of nights"
        }
    },
    "required": ["destination", "start_date", "stay_duration"],
    "additionalProperties": False
}

response2 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "user", "content": trip_text}
    ],
    reasoning_effort="none",
    max_completion_tokens=200,
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "trip_info",
            "strict": True,
            "schema": schema
        }
    }
)

data = json.loads(response2.choices[0].message.content)

print("METHOD 2 - JSON SCHEMA:")
print(data)
print("Types:", {k: type(v).__name__ for k, v in data.items()})

METHOD 2 - JSON SCHEMA:
{'destination': 'Tokyo', 'start_date': '2026-06-12', 'stay_duration': 5}
Types: {'destination': 'str', 'start_date': 'str', 'stay_duration': 'int'}


I tested two methods for extracting information from text into JSON.

The first method used a regular prompt, while the second used a strict JSON schema. The schema method returned the correct fields and data types, including an integer for the number of nights.

I would prefer the schema method for a computer program because the output structure is more predictable. However, the schema doesn't guarantee that every value is factually correct.

#### 24. Create a structured table comparing different programming languages.

Write a prompt that generates a table in this format (example below) with accurate information.

| Language    | Use Case               | Advantages                     | Disadvantages                |
|-------------|------------------------|---------------------------------|------------------------------|
| Python      | Data Science, AI       | Easy to learn, versatile       | Slower execution speed       |
| Java        | Enterprise Applications| Platform-independent, robust   | Verbose syntax               |
| JavaScript  | Web Development        | Fast, widely supported         | Limited for backend tasks    |

**Tip**: To enhance readability, you can copy and paste the generated table into a Markdown cell.

In [43]:
prompt = """
Create a Markdown table comparing Python, Java, and JavaScript.

Use exactly four columns:
Language | Use Case | Advantages | Disadvantages

Requirements:
- Include exactly three rows, one for each language.
- Give accurate and practical use cases.
- Include the main advantages and disadvantages.
- Keep each cell concise.
- Return only the Markdown table.
"""

response = ask(prompt, max_tokens=400)

table_text = response.choices[0].message.content
print(table_text)

| Language | Use Case | Advantages | Disadvantages |
|---|---|---|---|
| Python | Data science, automation, web backends | Easy to read, large ecosystem, rapid development | Slower runtime, weaker mobile/browser support |
| Java | Enterprise apps, Android, large backend systems | Strong performance, portable, robust tooling | Verbose syntax, slower startup, higher memory use |
| JavaScript | Web frontends, Node.js backends, full-stack apps | Runs in browsers, flexible, huge ecosystem | Dynamic typing can cause bugs, async complexity, ecosystem fragmentation |


In [44]:
from IPython.display import display, Markdown

display(Markdown(table_text))

| Language | Use Case | Advantages | Disadvantages |
|---|---|---|---|
| Python | Data science, automation, web backends | Easy to read, large ecosystem, rapid development | Slower runtime, weaker mobile/browser support |
| Java | Enterprise apps, Android, large backend systems | Strong performance, portable, robust tooling | Verbose syntax, slower startup, higher memory use |
| JavaScript | Web frontends, Node.js backends, full-stack apps | Runs in browsers, flexible, huge ecosystem | Dynamic typing can cause bugs, async complexity, ecosystem fragmentation |

The model generated a well-structured Markdown table comparing Python, Java, and JavaScript.

It included all four requested columns and gave relevant use cases, advantages, and disadvantages for each language.

I found the table format useful because it makes comparing different programming languages much easier.

#### 25. Generate a quiz about machine learning basics.

Look at the example below and write a prompt that generates a quiz with identical formatting. Ensure the title formatting, list markers (a, b, c, d), and the correct answer indicator (*) are accurately replicated.

**Machine learning basics quiz**

1. Which of the following best describes supervised learning?  
a) The model learns from unlabeled data to find hidden patterns  
b) The model learns from labeled data to make predictions *  
c) The model is programmed with explicit rules  
d) The model randomly guesses outputs without training

2. Which of these is a common application of machine learning in everyday life?  
a) Sorting files manually on a computer  
b) Email spam filtering *  
c) Writing code by hand  
d) Using a calculator for arithmetic

3. What is overfitting in machine learning?  
a) When the model performs well on new, unseen data  
b) When the model is too simple and underperforms  
c) When the model learns the training data too well, including noise, reducing generalization *  
d) When the model runs faster than expected

4. Which ethical concern is commonly associated with machine learning algorithms?  
a) Increased hardware costs  
b) Bias in training data leading to unfair predictions *  
c) The inability to process large datasets  
d) Lack of available programming languages

5. Which of the following is NOT a type of machine learning?  
a) Supervised learning  
b) Reinforcement learning  
c) Unsupervised learning  
d) Manual learning *

In [46]:
prompt = """
Create an original quiz about machine learning basics.

Follow this exact format:

**Machine learning basics quiz**

1. Question?
a) Answer
b) Answer *
c) Answer
d) Answer

Requirements:
- Create exactly 5 new questions.
- Each question must have 4 options: a, b, c, d.
- Mark exactly one correct answer per question with *.
- Keep the title and formatting exactly as shown.
- Make questions practical and scenario-based.
- Use these topics:
  1. Training vs testing data
  2. Regression
  3. Class imbalance
  4. Data leakage
  5. Feature scaling
- Do not use questions about supervised learning definitions,
  spam detection, overfitting, or machine learning types.
- Invent your own situations and answer options.
- Return only the quiz.
"""

response = ask(prompt, max_tokens=900)

quiz = response.choices[0].message.content
print(quiz)

**Machine learning basics quiz**

1. A company builds a model to estimate house prices using past sales records. Which dataset should be used to tune the model after training, before final evaluation?
a) The same records used to fit the model
b) A separate testing set only
c) A validation set split from the training data *
d) Random internet examples

2. A store wants to predict next month’s sales amount in dollars. Which model output best matches this goal?
a) A category such as “high,” “medium,” or “low”
b) A numeric value like 48,250 *
c) A yes/no answer
d) A list of product names

3. A medical app has 9,800 healthy cases and 200 rare-condition cases. What is the best immediate concern?
a) The model may mostly predict the healthy class and miss rare cases *
b) The model will always overfit because there are too many examples
c) The labels must be converted into numbers
d) The dataset is too large to train on

4. A team builds a loan-risk model and accidentally includes each applican

In [47]:
import re

questions = re.findall(r"(?m)^\d+\.\s", quiz)
options = re.findall(r"(?m)^[abcd]\)\s", quiz)
correct = re.findall(r"(?m)^[abcd]\).*?\*\s*$", quiz)

print("Questions:", len(questions))
print("Answer options:", len(options))
print("Marked correct answers:", len(correct))
print("Format valid:",
      len(questions) == 5 and
      len(options) == 20 and
      len(correct) == 5)

Questions: 5
Answer options: 20
Marked correct answers: 5
Format valid: True


#### 26. Create a comprehensive online course outline for beginners.

Write a prompt that generates entire course outline, following the example provided below. Ensure the prompt is detailed enough to cover all sections, with clear formatting and practical guidance. 

Sections to include in course outline:
- Course overview - description, objectives, target audience.
- Module breakdown - 5 modules with title, short description, three key learning outcomes and an interactive assignment in the end of each module.
- Assessment plan - a table outlining the assessment methods with description and weighting.
- Certification criteria - requirements to earn a certificate.
- Marketing strategy - step-by-step guide for promoting the course.

Include a detailed description of the type of information each section should contain. Maintain the **same formatting**, **section titles**, and **course name**. While the generated descriptions, modules and marketing strategy steps may differ and incorporate new ideas and concepts, it must remain aligned with the theme of the task.

Put the final output of your API call in a Markdown cell. 

#### Example:
---
---
## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-friendly course is designed to equip learners with the foundational skills and knowledge required to craft effective prompts for AI applications. By mastering prompt engineering, students will learn how to optimize AI models for various tasks, including text generation, data analysis, and creative problem-solving. The course is ideal for individuals interested in AI, developers, data scientists, and anyone looking to leverage AI tools in their personal or professional projects.

#### Objectives
- Understand the principles of prompt engineering and its role in AI workflows.
- Learn techniques to design, test, and refine prompts for optimal AI performance.
- Apply prompt engineering skills to real-world scenarios and projects.

#### Target Audience
- Beginners with no prior experience in AI or programming.
- Professionals seeking to integrate AI into their workflows.
- Students and enthusiasts curious about AI capabilities.

---

### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module introduces the concept of prompt engineering, its importance in AI applications, and the basics of interacting with AI models.
- **Key Learning Outcomes:**
  1. Define prompt engineering and its significance in AI workflows.
  2. Understand the structure and components of effective prompts.
  3. Explore real-world use cases of prompt engineering.
- **Interactive Assignment:** Write a simple prompt for a text-generation AI model to generate a short story.

---

#### Module 2: Principles of Effective Prompt Design
- **Description:** Learn the core principles and techniques for designing prompts that yield accurate and relevant AI outputs.
- **Key Learning Outcomes:**
  1. Identify factors that influence AI responses (e.g., context, specificity, tone).
  2. Explore strategies for iterative prompt refinement.
  3. Understand common pitfalls and how to avoid them.
- **Interactive Assignment:** Design and test three variations of a prompt for a chatbot to answer customer queries effectively.

---

#### Module 3: Advanced Techniques in Prompt Engineering
- **Description:** Dive deeper into advanced methods, including chaining prompts, dynamic prompts, and leveraging AI tools for complex tasks.
- **Key Learning Outcomes:**
  1. Learn how to chain prompts for multi-step tasks.
  2. Explore dynamic prompt techniques for adaptive AI responses.
  3. Understand how to integrate external data sources into prompts.
- **Interactive Assignment:** Create a multi-step prompt to generate a detailed report based on user-provided data.

---

#### Module 4: Applications of Prompt Engineering Across Industries
- **Description:** Discover how prompt engineering is applied in various industries, such as healthcare, education, marketing, and software development.
- **Key Learning Outcomes:**
  1. Analyze industry-specific use cases of prompt engineering.
  2. Learn how to tailor prompts for different domains and objectives.
  3. Explore ethical considerations in prompt engineering.
- **Interactive Assignment:** Develop a prompt for a marketing campaign, including product descriptions and social media posts.

---

#### Module 5: Building and Testing AI Projects with Prompt Engineering
- **Description:** Apply your skills to design, test, and deploy AI projects using prompt engineering techniques.
- **Key Learning Outcomes:**
  1. Plan and execute an AI project using prompt engineering.
  2. Test and refine prompts for optimal project outcomes.
  3. Document and present your project effectively.
- **Interactive Assignment:** Create a mini-project using prompt engineering to solve a real-world problem (e.g., automating customer support).

---

### Assessment Plan

| Assessment Method      | Description                          | Weightage (%) |
|------------------------|--------------------------------------|---------------|
| Quizzes               | Multiple-choice quizzes after each module | 30%          |
| Assignments           | Practical exercises for each module  | 40%           |
| Final Project         | A comprehensive project combining all skills | 20%          |
| Participation         | Active participation in discussions  | 10%           |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve a minimum score of 70% across all assessments.
2. Submit all interactive assignments and the final project.
3. Actively participate in at least 80% of discussion forums and live sessions.

---

### Marketing Strategy

#### Step 1: Social Media Campaigns
- Create engaging posts highlighting the course benefits, modules, and outcomes.
- Share testimonials and success stories from beta testers or early participants.
- Use targeted ads on platforms like LinkedIn, Facebook, and Instagram.

#### Step 2: Email Marketing
- Build an email list of potential learners (e.g., professionals, students).
- Send a series of emails introducing the course, its benefits, and enrollment deadlines.
- Include exclusive early-bird discounts and referral incentives.

#### Step 3: Partnerships
- Collaborate with universities, coding bootcamps, and AI communities to promote the course.
- Partner with influencers and thought leaders in the AI space for endorsements.

#### Step 4: Content Creation
- Publish blog posts, videos, and webinars on topics related to prompt engineering.
- Share free resources (e.g., mini-guides, templates) to attract interest.
- Host live Q&A sessions to engage with the audience and answer questions about the course.

#### Step 5: Testimonials and Reviews
- Collect feedback from early participants and showcase their success stories.
- Encourage learners to share their experiences on social media and review platforms.

---

In [52]:
from pathlib import Path
import re

prompt = """
Create an original, comprehensive online course outline for beginners.

Use this EXACT title:
## Course Outline: Mastering Prompt Engineering for AI Applications

Keep the following EXACT main headings, without numbering them:
### Course Overview
### Module Breakdown
### Assessment Plan
### Certification Criteria
### Marketing Strategy

Follow these requirements:

SECTION 1 — Course Overview
- Write a clear course description for beginners.
- Add #### Objectives with exactly 3 learning objectives.
- Add #### Target Audience with 3 groups of learners.

SECTION 2 — Module Breakdown
Create exactly 5 modules, using this structure:

#### Module 1: [Title]
- **Description:** Explain the module content.
- **Key Learning Outcomes:**
  1. First learning outcome.
  2. Second learning outcome.
  3. Third learning outcome.
- **Interactive Assignment:** Describe a practical exercise.

Repeat this format for all 5 modules.

The modules should cover:
1. Introduction to Prompt Engineering
2. Effective Prompt Design
3. Advanced Prompting Techniques
4. Real-World Applications
5. Building and Testing an AI Project

Create original descriptions, learning outcomes and assignments.

SECTION 3 — Assessment Plan
Create a Markdown table with exactly these columns:
Assessment Method | Description | Weightage (%)

Include:
- Quizzes — 30%
- Assignments — 40%
- Final Project — 20%
- Participation — 10%

SECTION 4 — Certification Criteria
Explain these requirements:
1. Minimum overall score of 70%.
2. Completion of all assignments and the final project.
3. Participation in at least 80% of course activities.

SECTION 5 — Marketing Strategy
Create exactly these headings:

#### Step 1: Social Media Campaigns
#### Step 2: Email Marketing
#### Step 3: Partnerships
#### Step 4: Content Creation
#### Step 5: Testimonials and Reviews

Give at least 2 practical actions for each step.
Do not invent testimonials from real students.

FINAL FORMATTING RULES:
- Keep the exact course title and section headings.
- Use Markdown formatting.
- Separate major sections with ---.
- Use bullet points and numbered learning outcomes.
- Keep all sections in the required order.
- Do not copy the wording of the sample course.
- Do not add introductions or explanations outside the outline.
- Do not wrap the result in a code block.
"""

response = ask(prompt, max_tokens=4500)
course = response.choices[0].message.content

for i, section in enumerate([
    "Course Overview",
    "Module Breakdown",
    "Assessment Plan",
    "Certification Criteria",
    "Marketing Strategy"
], 1):
    course = course.replace(f"### {i}. {section}", f"### {section}")

sections = [
    "### Course Overview",
    "### Module Breakdown",
    "### Assessment Plan",
    "### Certification Criteria",
    "### Marketing Strategy"
]

modules = re.findall(r"(?m)^#### Module \d+:", course)
steps = re.findall(r"(?m)^#### Step \d+:", course)

print("Modules:", len(modules), "/ 5")
print("Marketing steps:", len(steps), "/ 5")
print("All sections present:", all(s in course for s in sections))
print("Output complete:", response.choices[0].finish_reason == "stop")

file_path = Path("course_outline.md")
file_path.write_text(course, encoding="utf-8")

print("Saved to:", file_path.resolve())

Modules: 5 / 5
Marketing steps: 5 / 5
All sections present: True
Output complete: True
Saved to: C:\Users\jelisejs.bumazkins\Documents\GitHub\Bootcamp_Accenture_JB\day 8\course_outline.md


## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview

This beginner-friendly course introduces learners to the fundamentals of prompt engineering and shows how to communicate effectively with AI tools to produce better results. Participants will learn how to write clear prompts, improve AI responses through iteration, and apply prompting techniques to practical use cases across different industries. By the end of the course, learners will be able to design, test, and refine prompts for everyday and project-based AI applications.

#### Objectives
1. Understand the core principles of prompt engineering and how AI systems interpret instructions.
2. Create effective prompts that improve clarity, relevance, and output quality.
3. Apply prompting strategies to real-world tasks and simple AI-driven projects.

#### Target Audience
- Beginners who want to learn how to use AI tools more effectively.
- Students and professionals exploring AI for productivity, research, or creative work.
- Entrepreneurs and content creators looking to integrate AI into their workflows.

---

### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module introduces the purpose of prompt engineering, how AI language models respond to input, and why prompt quality matters. Learners will explore the relationship between instruction, context, and AI output while gaining a simple framework for writing prompts.
- **Key Learning Outcomes:**
  1. Define prompt engineering and explain its role in AI applications.
  2. Identify the main components of a basic prompt.
  3. Recognize common mistakes that lead to weak AI responses.
- **Interactive Assignment:** Write three prompts for the same task—one vague, one improved, and one highly specific—and compare the differences in AI responses.

#### Module 2: Effective Prompt Design
- **Description:** This module focuses on crafting prompts that generate clear, useful, and consistent results. Learners will practice adding context, constraints, examples, and desired formats to improve the quality of responses.
- **Key Learning Outcomes:**
  1. Build prompts using structure, context, and instruction.
  2. Use formatting cues to shape the AI’s output.
  3. Refine prompts through revision and testing.
- **Interactive Assignment:** Design a prompt template for a common work task, then test and revise it to produce a more accurate and organized response.

#### Module 3: Advanced Prompting Techniques
- **Description:** Learners will explore more sophisticated prompting methods such as role-based prompting, multi-step instructions, and iterative refinement. The module also introduces techniques for controlling tone, depth, and response style.
- **Key Learning Outcomes:**
  1. Apply role-based prompts to guide AI behavior and perspective.
  2. Use step-by-step prompting to improve complex outputs.
  3. Adjust prompts to control tone, detail level, and response structure.
- **Interactive Assignment:** Create a multi-step prompt that asks the AI to analyze a problem, generate options, and recommend a final answer in a chosen format.

#### Module 4: Real-World Applications
- **Description:** This module demonstrates how prompt engineering can be used in everyday and professional settings. Learners will examine practical examples in writing, research, brainstorming, customer support, education, and planning.
- **Key Learning Outcomes:**
  1. Match prompting techniques to specific real-world use cases.
  2. Adapt prompts for different professional goals and audiences.
  3. Evaluate the usefulness of AI outputs in applied settings.
- **Interactive Assignment:** Choose one real-world scenario, such as marketing copy, lesson planning, or meeting notes, and create a prompt that generates a usable first draft.

#### Module 5: Building and Testing an AI Project
- **Description:** In this final module, learners bring together all course concepts to create a small AI-based project. They will define a use case, design prompts, test outputs, improve performance, and document their process.
- **Key Learning Outcomes:**
  1. Plan a basic AI project with a clear purpose and workflow.
  2. Develop and test prompts to support project goals.
  3. Assess results and make improvements based on feedback.
- **Interactive Assignment:** Build a mini prompt-based AI project, test it with multiple inputs, and submit a short summary explaining what worked, what changed, and why.

---

### Assessment Plan

| Assessment Method | Description | Weightage (%) |
|---|---|---:|
| Quizzes | Short quizzes to check understanding of core concepts and terminology. | 30% |
| Assignments | Practical prompt-writing tasks and module-based exercises. | 40% |
| Final Project | A small AI project that demonstrates prompt design, testing, and refinement. | 20% |
| Participation | Engagement in discussions, peer feedback, and course activities. | 10% |

---

### Certification Criteria

To earn certification, learners must meet all of the following requirements:

1. Achieve a minimum overall score of 70%.
2. Complete all assignments and the final project.
3. Participate in at least 80% of course activities.

---

### Marketing Strategy

#### Step 1: Social Media Campaigns
- Share short before-and-after prompt examples on platforms like LinkedIn, Instagram, and X to show the value of prompt improvement.
- Post weekly tips, mini-challenges, and carousel graphics that highlight practical AI prompting skills.

#### Step 2: Email Marketing
- Send a beginner-focused welcome sequence that explains the course benefits, learning outcomes, and key takeaways.
- Create a segmented email campaign for students, professionals, and creators with tailored examples of how prompt engineering can help them.

#### Step 3: Partnerships
- Collaborate with tech communities, online learning platforms, and productivity influencers to expand course visibility.
- Offer group enrollment options or affiliate partnerships with organizations interested in AI upskilling.

#### Step 4: Content Creation
- Publish blog articles, short tutorial videos, and downloadable prompt templates that showcase real course concepts.
- Produce a free sample lesson or webinar to give potential learners a preview of the learning experience.

#### Step 5: Testimonials and Reviews
- Request feedback from pilot learners or early participants to refine course messaging and highlight the course’s practical value.
- Feature verified learner reviews, instructor insights, or partner endorsements without fabricating personal testimonials.

####